In [2]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath(".."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

ticker = 'QQQ'
[4, 5, 6, 8, 10, 15, 20, 25, 30]
lb = 8
cat_cols_all = tags[(tags['Type'] == 'Raw')]['Indicator'].tolist()

def extract(ticker, returns, lb, cat_cols_all, windows=[10, 25]):
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

#df = extract(ticker, returns, lb, cat_cols_all)

# Start with must include columns (slope)

In [22]:
def print_metrics(metrics):
        for thresh, metric_values in metrics.items():
            print(f"  Threshold {thresh}: {metric_values}")

def optimize_tests(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_test, y_train, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                metrics[t] = {
                    'PosF1': round(f1_score(y_test_valid, y_pred_valid), 3),
                    'NegF1': round((2*round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3)*round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3))/(round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3) + round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3)),3),
                    "PosTot": round(postot, 0),  # % of total positives identified
                    'PosPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=1), 3),  # % of positive predictions that were actually positive
                    "NegTot": round(negtot, 0),  # % of total negatives identified
                    'NegPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3),  # % of negative predictions that were actually negative
                    'PosCnt': sum(y_pred_valid == 1),
                    'NegCnt': sum(y_pred_valid == 0),
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [8, 10, 12], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }

    if date == 'lag':
        p = 350
        df_ind_rec = df_indicators.iloc[:p].copy()
        df_pred_rec = df_predict.iloc[:p].copy()
        df_indicators = df_indicators.iloc[p:]
        df_predict = df_predict.iloc[p:]

        
        # Split data once
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.10, random_state=42, shuffle=True)
        X_test = pd.concat([X_test, df_ind_rec], axis=0)
        y_test = pd.concat([y_test, df_pred_rec], axis=0)
        
        """
        X_test = df_indicators.iloc[:p].copy()
        y_test = df_predict.iloc[:p].copy()
        X_train = df_indicators.iloc[p:].copy()
        y_train = df_predict.iloc[p:].copy()
        
        print(f'xte {len(X_test)} | xtr{len(X_train)} | yte{len(y_test)} | ytr{len(y_train)}')
        """

    else:
        
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=None, shuffle=True)

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_test, y_train, y_test, opt, thresh)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
    
def optimize_ttv(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                metrics[t] = {
                    'TT_Len': len(df_indicators),
                    #'PosF1': round(f1_score(y_test_valid, y_pred_valid), 3),
                    #'NegF1': round((2*round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3)*round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3))/(round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3) + round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3)),3),
                    "PosTot": round(postot, 0),  # % of total positives identified
                    "NegTot": round(negtot, 0),  # % of total negatives identified
                    'PosPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 3),  # % of positive predictions that were actually positive
                    'NegPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 3),  # % of negative predictions that were actually negative
                    'PosCnt': sum(y_pred_valid == 1),
                    'NegCnt': sum(y_pred_valid == 0),
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [14], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }

    if date == 'lag':
        
        p = 100
        X_test = df_indicators.iloc[:p].copy()
        y_test = df_predict.iloc[:p].copy()
        df_indicators = df_indicators.iloc[p:].copy()
        df_predict = df_predict.iloc[p:].copy()

        # Split data once
        #X_train, X_val, y_train, y_val = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=42, shuffle=False)
        
        p=250
        X_val = df_indicators.iloc[:p].copy()
        y_val = df_predict.iloc[:p].copy()
        X_train = df_indicators.iloc[p:].copy()
        y_train = df_predict.iloc[p:].copy()
        """
        print(f'xte {len(X_test)} | xtr{len(X_train)} | yte{len(y_test)} | ytr{len(y_train)}')
        """

    else:
        
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=None, shuffle=True)

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
    
def optimize_ttv2(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size, val_size, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 3)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 3)
                poscnt = sum(y_pred_valid == 1)
                negcnt = sum(y_pred_valid == 0)
                metrics[t] = {
                    'TT_Len': len(df_indicators),
                    "PosTot": postot,  
                    "NegTot": negtot,  
                    "PosID": round(posprec * poscnt / postot, 3),
                    'PosPrec': posprec,  
                    "NegID":round(negprec * negcnt / negtot, 3),
                    'NegPrec': negprec,  
                    'PosCnt': poscnt,
                    'NegCnt': negcnt,
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [8, 10, 12], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }
        
    X_test = df_indicators.iloc[:test_size].copy()
    y_test = df_predict.iloc[:test_size].copy()
    df_indicators = df_indicators.iloc[test_size:].copy()
    df_predict = df_predict.iloc[test_size:].copy()

    X_val = df_indicators.iloc[:val_size].copy()
    y_val = df_predict.iloc[:val_size].copy()
    X_train = df_indicators.iloc[val_size:].copy()
    y_train = df_predict.iloc[val_size:].copy()

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
   

# Category based groupings

In [4]:
raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()
raw_duration = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration')]['Indicator'].tolist()
raw_trend = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend')]['Indicator'].tolist()
raw_trend_ratio = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio')]['Indicator'].tolist()
raw_volatility = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility')]['Indicator'].tolist()
raw_momentum = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum')]['Indicator'].tolist()

raw_cols_list = [raw_all, raw_duration, raw_trend, raw_trend_ratio, raw_volatility, raw_momentum]
raw_names = ['all', 'duration', 'trend', 'trend_ratio', 'volatility', 'momentum']

raw_cols_list = [raw_duration, raw_trend, raw_trend_ratio, raw_volatility, raw_momentum]
raw_names = ['duration', 'trend', 'trend_ratio', 'volatility', 'momentum']

In [134]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.55]
arch_types = ['deep', 'moderate', 'shallow']
dates = ['lag']#, 'lag']
n = len(raw_cols_list)
results = []

for ticker in tickers:
    
    if ticker == 'QQQ':
        dates = ['lag']#, 'current']
    else:
        dates = ['lag']

    for date in dates:

        if ticker == 'QQQ':
                returns = [5, 10, 20, 30]#[4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
        else:
            returns = [4, 6, 8, 10, 15]

        for r in returns:

            # loop r = 1 .. 5
            for i in range(1, n+1):
                
                # all index-combos of length r
                for idx_combo in itertools.combinations(range(n), i):

                    # pull out the names and the column‐lists
                    name_combo = [ raw_names[i]    for i in idx_combo ]
                    cols_combo = [ raw_cols_list[i] for i in idx_combo ]
                    
                    # flatten the list of lists into one feature list
                    cols = [ feat for sub in cols_combo for feat in sub ]
                    
                    # build a descriptive name, e.g. "trend_volatility_momentum"
                    combo_name = "_".join(name_combo)

                    df_ph = df.copy()
                    df_ph = df_ph.iloc[r:].copy()
                    return_col = f"Return_{r}"
                    model_key = f"QQQ_{r}"
                    counts = df_ph[return_col].value_counts()
                    neg = counts.get(0, 0)
                    pos = counts.get(1, 1)  # prevent division by zero
                    scale_pos_weight = neg / pos
                    imbalance_ratio = min(pos, neg) / max(pos, neg)
                    min_child_weight = max(int(max(1, round(10 * imbalance_ratio))),5)

                    for arch in arch_types:

                        depth = min(int(math.floor(len(cols) / 2)), 10)
                        depth = min(int(round(len(cols) / 2, 0)), 10)
                        depth = max(depth, 6) #minimum depth set to 6 if below 6

                        # Choose evaluation metric
                        opt = 'matthews_corrcoef'

                        # Combine features with return column, drop missing
                        #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                        used_cols = cols + [return_col]
                        df_model = df_ph[used_cols].dropna()
                        #print(df_ph['Date'].iloc[0])
                        
                        df_indicators = df_model[cols]
                        df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                        df_predict = df_model[return_col]
                        
                        print(f"Results for {combo_name} | {arch} | {ticker}_{r} | {date}")
                        xg_metrics, best_xg_model = optimize_tests(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=True)
                        metrics = list(xg_metrics.values())[0]
                        # build one flat record
                        row = {'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'date': date, **metrics}
                        results.append(row)
                        print('---------------------------')

Results for duration | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.774, 'NegF1': 0.618, "PosID'd": 0.826, 'PosPrec': 0.728, "NegID'd": 0.558, 'NegPrec': 0.692, 'PosCnt': 294, 'NegCnt': 146}
---------------------------
Results for duration | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.743, 'NegF1': 0.576, "PosID'd": 0.771, 'PosPrec': 0.717, "NegID'd": 0.544, 'NegPrec': 0.613, 'PosCnt': 272, 'NegCnt': 150}
---------------------------
Results for duration | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.709, 'NegF1': 0.552, "PosID'd": 0.718, 'PosPrec': 0.7, "NegID'd": 0.541, 'NegPrec': 0.563, 'PosCnt': 240, 'NegCnt': 151}
---------------------------
Results for trend | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.716, 'NegF1': 0.549, "PosID'd": 0.719, 'PosPrec': 0.713, "NegID'd": 0.545, 'NegPrec': 0.553, 'PosCnt': 244, 'NegCnt': 152}
---------------------------
Results for trend | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.731, 'NegF1': 0.588, "PosID'd": 0

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.731, 'NegF1': 0.52, "PosID'd": 0.777, 'PosPrec': 0.689, "NegID'd": 0.47, 'NegPrec': 0.582, 'PosCnt': 309, 'NegCnt': 146}
---------------------------
Results for trend_momentum | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.718, 'NegF1': 0.52, "PosID'd": 0.757, 'PosPrec': 0.683, "NegID'd": 0.478, 'NegPrec': 0.57, 'PosCnt': 224, 'NegCnt': 114}
---------------------------
Results for trend_momentum | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.728, 'NegF1': 0.566, "PosID'd": 0.79, 'PosPrec': 0.675, "NegID'd": 0.503, 'NegPrec': 0.648, 'PosCnt': 240, 'NegCnt': 122}
---------------------------
Results for trend_ratio_volatility | deep | QQQ_5 | lag


/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.724, 'NegF1': 0.528, "PosID'd": 0.774, 'PosPrec': 0.68, "NegID'd": 0.476, 'NegPrec': 0.594, 'PosCnt': 272, 'NegCnt': 133}
---------------------------
Results for trend_ratio_volatility | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.716, 'NegF1': 0.538, "PosID'd": 0.726, 'PosPrec': 0.706, "NegID'd": 0.526, 'NegPrec': 0.55, 'PosCnt': 218, 'NegCnt': 129}
---------------------------
Results for trend_ratio_volatility | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.687, 'NegF1': 0.513, "PosID'd": 0.699, 'PosPrec': 0.676, "NegID'd": 0.5, 'NegPrec': 0.527, 'PosCnt': 182, 'NegCnt': 112}
---------------------------
Results for trend_ratio_momentum | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.74, 'NegF1': 0.539, "PosID'd": 0.788, 'PosPrec': 0.697, "NegID'd": 0.486, 'NegPrec': 0.605, 'PosCnt': 310, 'NegCnt': 147}
---------------------------
Results for trend_ratio_momentum | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.744, 'NegF1': 0.552,

...

  Threshold 0.55: {'PosF1': 0.737, 'NegF1': 0.597, "PosID'd": 0.708, 'PosPrec': 0.768, "NegID'd": 0.637, 'NegPrec': 0.562, 'PosCnt': 246, 'NegCnt': 178}
---------------------------
Results for duration_trend_ratio_volatility | shallow | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.735, 'NegF1': 0.561, "PosID'd": 0.685, 'PosPrec': 0.794, "NegID'd": 0.64, 'NegPrec': 0.5, 'PosCnt': 238, 'NegCnt': 174}
---------------------------
Results for duration_trend_ratio_momentum | deep | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.78, 'NegF1': 0.65, "PosID'd": 0.743, 'PosPrec': 0.821, "NegID'd": 0.706, 'NegPrec': 0.602, 'PosCnt': 296, 'NegCnt': 211}
---------------------------
Results for duration_trend_ratio_momentum | moderate | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.746, 'NegF1': 0.613, "PosID'd": 0.689, 'PosPrec': 0.813, "NegID'd": 0.701, 'NegPrec': 0.545, 'PosCnt': 262, 'NegCnt': 211}
---------------------------
Results for duration_trend_ratio_momentum | shallow | QQQ_10 | lag
  Threshold 0

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.751, 'NegF1': 0.548, "PosID'd": 0.711, 'PosPrec': 0.796, "NegID'd": 0.611, 'NegPrec': 0.497, 'PosCnt': 284, 'NegCnt': 183}
---------------------------
Results for trend_trend_ratio_volatility | moderate | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.722, 'NegF1': 0.572, "PosID'd": 0.679, 'PosPrec': 0.77, "NegID'd": 0.633, 'NegPrec': 0.522, 'PosCnt': 239, 'NegCnt': 182}
---------------------------
Results for trend_trend_ratio_volatility | shallow | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.724, 'NegF1': 0.539, "PosID'd": 0.679, 'PosPrec': 0.775, "NegID'd": 0.606, 'NegPrec': 0.485, 'PosCnt': 240, 'NegCnt': 171}
---------------------------
Results for trend_trend_ratio_momentum | deep | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.776, 'NegF1': 0.633, "PosID'd": 0.729, 'PosPrec': 0.83, "NegID'd": 0.708, 'NegPrec': 0.573, 'PosCnt': 276, 'NegCnt': 199}
---------------------------
Results for trend_trend_ratio_momentum | moderate | QQQ_10 | lag
  Threshold 0.55: {

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.781, 'NegF1': 0.599, "PosID'd": 0.72, 'PosPrec': 0.852, "NegID'd": 0.708, 'NegPrec': 0.519, 'PosCnt': 305, 'NegCnt': 210}
---------------------------
Results for duration_trend_trend_ratio_volatility_momentum | shallow | QQQ_20 | lag
  Threshold 0.55: {'PosF1': 0.776, 'NegF1': 0.632, "PosID'd": 0.706, 'PosPrec': 0.861, "NegID'd": 0.755, 'NegPrec': 0.544, 'PosCnt': 273, 'NegCnt': 215}
---------------------------
Results for duration | deep | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.814, 'NegF1': 0.525, "PosID'd": 0.842, 'PosPrec': 0.788, "NegID'd": 0.483, 'NegPrec': 0.574, 'PosCnt': 353, 'NegCnt': 122}
---------------------------
Results for duration | moderate | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.76, 'NegF1': 0.47, "PosID'd": 0.769, 'PosPrec': 0.752, "NegID'd": 0.458, 'NegPrec': 0.482, 'PosCnt': 314, 'NegCnt': 137}
---------------------------
Results for duration | shallow | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.758, 'NegF1': 0.481, "PosID'd

# Velocity based groupings

In [5]:
cat_slow = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
cat_moderate = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
cat_fast = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

"""
cat_slows = cat_slow + ['Close_slope10', 'Close_slope25', 'Close_slope50']
cat_moderates = cat_moderate + ['Close_slope10', 'Close_slope25', 'Close_slope50']
cat_fasts = cat_fast + ['Close_slope10', 'Close_slope25', 'Close_slope50']
"""

windows = [25]
# EMA column‐name lists
cat_slow25      = [f"{col}_EMA{w}" for col in cat_slow     for w in windows]
cat_moderate25  = [f"{col}_EMA{w}" for col in cat_moderate for w in windows]
cat_fast25      = [f"{col}_EMA{w}" for col in cat_fast     for w in windows]

windows = [10]
# EMA column‐name lists
cat_slow10      = [f"{col}_EMA{w}" for col in cat_slow     for w in windows]
cat_moderate10  = [f"{col}_EMA{w}" for col in cat_moderate for w in windows]
cat_fast10      = [f"{col}_EMA{w}" for col in cat_fast     for w in windows]

raw_cols_list = [cat_slow10, cat_moderate10, cat_fast10]
raw_names = ['cat_slow', 'cat_moderate', 'cat_fast']

raw_cols_list = [cat_slow, cat_slow25, cat_slow10, cat_moderate, cat_moderate25, 
                 cat_moderate10, cat_fast, cat_fast25, cat_fast10]
raw_names = ['cat_slow', 'cat_slow25', 'cat_slow10', 'cat_moderate', 'cat_moderate25', 
             'cat_moderate25', 'cat_fast', 'cat_fast25', 'cat_fast25']

# Velocity x Category Combo Groupings

In [6]:
# By category × velocity
raw_duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

raw_trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

raw_trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

raw_volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

raw_momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

In [27]:
import itertools

# 1) Map your 15 groups to their feature‐lists
group_cols = {
    'duration_slow': raw_duration_slow,
    'duration_moderate': raw_duration_moderate,
    'duration_fast': raw_duration_fast,
    'trend_slow': raw_trend_slow,
    'trend_moderate': raw_trend_moderate,
    'trend_fast': raw_trend_fast,
    'trend_ratio_slow': raw_trend_ratio_slow,
    'trend_ratio_moderate': raw_trend_ratio_moderate,
    'trend_ratio_fast': raw_trend_ratio_fast,
    'volatility_slow': raw_volatility_slow,
    'volatility_moderate': raw_volatility_moderate,
    'volatility_fast': raw_volatility_fast,
    'momentum_slow': raw_momentum_slow,
    'momentum_moderate': raw_momentum_moderate,
    'momentum_fast': raw_momentum_fast,
}

group_names = list(group_cols.keys())

# 2) Generate all 5‐combos, filtering out any with >2 from one category
valid_combos = []
for combo in itertools.combinations(group_names, 3):
    # extract the category portion (everything before the last "_")
    cats = [name.rsplit('_', 1)[0] for name in combo]
    # ensure no category appears more than twice
    if all(cats.count(cat) <= 2 for cat in set(cats)):
        valid_combos.append(combo)

# 3) (Optional) Build a mapping from combo → flattened feature list
combos_features = {
    combo: [feat for grp in combo for feat in group_cols[grp]]
    for combo in valid_combos
}

print(len(combos_features))

450


In [29]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.5]
arch_types = ['deep']#['deep', 'moderate', 'shallow']
n = len(raw_cols_list)
results = []
returns = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
name_to_combo = { "_".join(c): c for c in valid_combos }
df = extract(ticker, returns, lb, raw_all)

for ticker in tickers:

    if ticker == 'QQQ':
            returns = [1, 2, 3, 4, 5, 6, 8, 10]#[4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
    else:
        returns = [4, 6, 8, 10, 15]

    for r in returns:
            
        for combo in valid_combos:

            combo_name = "_".join(combo)
            cols = combos_features[combo]
            
            #df = extract(ticker, returns, lb, cat_cols_all, windows=[10, 25])
            df_ph = df.copy()
            df_ph = df_ph.iloc[r:].copy()
            return_col = f"Return_{r}"
            model_key = f"QQQ_{r}"
            counts = df_ph[return_col].value_counts()
            neg = counts.get(0, 0)
            pos = counts.get(1, 1)  # prevent division by zero
            scale_pos_weight = neg / pos

            for arch in arch_types:

                # Choose evaluation metric
                opt = 'matthews_corrcoef'

                # Combine features with return column, drop missing
                #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                used_cols = cols + [return_col]
                df_model = df_ph[used_cols].dropna()
                #print(df_ph['Date'].iloc[0])
                
                df_indicators = df_model[cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = df_model[return_col]
                
                print(f"Results for {combo_name} | {arch} | {ticker}_{r}")
                xg_metrics, best_xg_model = optimize_ttv2(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size=100, val_size=400, return_metrics=True)
                metrics = list(xg_metrics.values())[0]
                # build one flat record
                row = {'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': 400, **metrics}
                results.append(row)
                """
                xg_metrics, best_xg_model = optimize_ttv2(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size=100, val_size=400, return_metrics=True)
                row = {'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': 400, **metrics}
                results.append(row)
                """
                print('---------------------------')

Results for duration_slow_duration_moderate_trend_slow | deep | QQQ_1
  Threshold 0.5: {'TT_Len': 1657, 'PosTot': 56, 'NegTot': 44, 'PosID': 0.536, 'PosPrec': 0.588, 'NegID': 0.522, 'NegPrec': 0.469, 'PosCnt': 51, 'NegCnt': 49}
  Threshold 0.5: {'TT_Len': 1657, 'PosTot': 56, 'NegTot': 44, 'PosID': 0.25, 'PosPrec': 0.56, 'NegID': 0.75, 'NegPrec': 0.44, 'PosCnt': 25, 'NegCnt': 75}
---------------------------
Results for duration_slow_duration_moderate_trend_moderate | deep | QQQ_1
  Threshold 0.5: {'TT_Len': 1657, 'PosTot': 56, 'NegTot': 44, 'PosID': 0.518, 'PosPrec': 0.604, 'NegID': 0.568, 'NegPrec': 0.481, 'PosCnt': 48, 'NegCnt': 52}
  Threshold 0.5: {'TT_Len': 1657, 'PosTot': 56, 'NegTot': 44, 'PosID': 0.393, 'PosPrec': 0.595, 'NegID': 0.659, 'NegPrec': 0.46, 'PosCnt': 37, 'NegCnt': 63}
---------------------------
Results for duration_slow_duration_moderate_trend_fast | deep | QQQ_1
  Threshold 0.5: {'TT_Len': 1657, 'PosTot': 56, 'NegTot': 44, 'PosID': 0.446, 'PosPrec': 0.568, 'NegID'

KeyboardInterrupt: 

In [30]:
results_df = pd.DataFrame(results)
results_df.to_csv('val_test.csv')